# Notebook 01 — Data Ingestion & Knowledge Population

## Objective

This notebook builds the first stage of the Forge AI Engineering Intelligence Platform.

Its purpose is to automatically collect information from trusted sources defined in the Source Registry, clean and normalize the content, and prepare it for structured knowledge extraction.

The output of this notebook will be validated JSON records that populate the Forge Knowledge Base.

---

## Pipeline

Source Registry
→ Fetch Official Documentation
→ Parse Content
→ Clean & Normalize Text
→ Extract Metadata
→ Validate Against Schemas
→ Save to Knowledge Base

---

## Input

- Source Registry (`sources/`)
- Manifest (`manifest.json`)

## Output

- Structured Knowledge Base (`knowledge_base/`)


In [1]:
import os
import json
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from datetime import datetime
from IPython.display import display
import httpx

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!unzip -q "/content/drive/MyDrive/forge.zip" -d "/content/drive/MyDrive/"

In [4]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

print(PROJECT_ROOT.exists())

for item in sorted(PROJECT_ROOT.iterdir()):
    print(item.name)

True
.env.example
.git
.gitignore
LICENSE
README.md
ai-lab
ai_engine
app
backend
datasets
docker-compose.yml
docs
frontend
manifest.json
notebooks
requirements.txt
schemas
scratch
screenshots
scripts
sources
tests
tools


In [5]:
CONFIG = {
    "project_root": PROJECT_ROOT,
    "manifest": PROJECT_ROOT / "manifest.json",
    "sources": PROJECT_ROOT / "sources",
    "schemas": PROJECT_ROOT / "schemas",
    "knowledge": PROJECT_ROOT / "knowledge_base",
    "raw": PROJECT_ROOT / "knowledge_base" / "raw",
    "processed": PROJECT_ROOT / "knowledge_base" / "processed",
    "layer1": PROJECT_ROOT / "knowledge_base" / "layer_1",
    "layer2": PROJECT_ROOT / "knowledge_base" / "layer_2",
    "layer3": PROJECT_ROOT / "knowledge_base" / "layer_3",
    "layer4": PROJECT_ROOT / "knowledge_base" / "layer_4",
    "layer5": PROJECT_ROOT / "knowledge_base" / "layer_5",
    "chunks": PROJECT_ROOT / "knowledge_base" / "chunks",
    "embeddings": PROJECT_ROOT / "knowledge_base" / "embeddings",
    "logs": PROJECT_ROOT / "knowledge_base" / "logs"
}

In [6]:
for key, path in CONFIG.items():
    if isinstance(path, Path) and path.suffix == "":
        path.mkdir(parents=True, exist_ok=True)

print("Forge project initialized successfully.")

Forge project initialized successfully.


In [7]:
print("\nForge Configuration\n")

for key, value in CONFIG.items():
    print(f"{key:<12} -> {value}")


Forge Configuration

project_root -> /content/drive/MyDrive/forge
manifest     -> /content/drive/MyDrive/forge/manifest.json
sources      -> /content/drive/MyDrive/forge/sources
schemas      -> /content/drive/MyDrive/forge/schemas
knowledge    -> /content/drive/MyDrive/forge/knowledge_base
raw          -> /content/drive/MyDrive/forge/knowledge_base/raw
processed    -> /content/drive/MyDrive/forge/knowledge_base/processed
layer1       -> /content/drive/MyDrive/forge/knowledge_base/layer_1
layer2       -> /content/drive/MyDrive/forge/knowledge_base/layer_2
layer3       -> /content/drive/MyDrive/forge/knowledge_base/layer_3
layer4       -> /content/drive/MyDrive/forge/knowledge_base/layer_4
layer5       -> /content/drive/MyDrive/forge/knowledge_base/layer_5
chunks       -> /content/drive/MyDrive/forge/knowledge_base/chunks
embeddings   -> /content/drive/MyDrive/forge/knowledge_base/embeddings
logs         -> /content/drive/MyDrive/forge/knowledge_base/logs


In [8]:
with open(CONFIG["manifest"], "r", encoding="utf-8") as file:
    manifest = json.load(file)

print("Manifest loaded successfully.")

Manifest loaded successfully.


In [9]:
print(json.dumps(manifest, indent=4))

{
    "registry_version": "1.1",
    "schema_version": "1.0",
    "generated_at": "2026-07-28T09:29:30Z",
    "total_categories": 11,
    "total_sources": 117,
    "category_counts": {
        "chunking": 11,
        "deployment": 35,
        "embedding": 5,
        "evaluation": 8,
        "fine_tuning": 8,
        "framework": 4,
        "llm": 7,
        "prompting": 12,
        "rerankers": 9,
        "retrieval": 12,
        "vectordb": 6
    },
    "validation_status": "SUCCESS",
    "frozen": true
}


In [10]:
REQUIRED_FIELDS = [
    "registry_version",
    "schema_version",
    "generated_at",
    "total_categories",
    "total_sources",
    "category_counts",
    "validation_status",
    "frozen"
]

missing_fields = [field for field in REQUIRED_FIELDS if field not in manifest]

if missing_fields:
    raise ValueError(f"Missing required manifest fields: {missing_fields}")

print("Manifest structure validated successfully.")

Manifest structure validated successfully.


In [11]:
assert manifest["validation_status"] == "SUCCESS", "Manifest validation failed."
assert manifest["frozen"] is True, "Registry must be frozen before ingestion."

print("Manifest integrity checks passed.")

Manifest integrity checks passed.


In [12]:
print("\nForge Source Registry Summary\n")

print(f"Registry Version : {manifest['registry_version']}")
print(f"Schema Version   : {manifest['schema_version']}")
print(f"Generated At     : {manifest['generated_at']}")
print(f"Categories       : {manifest['total_categories']}")
print(f"Sources          : {manifest['total_sources']}")
print(f"Frozen           : {manifest['frozen']}")


Forge Source Registry Summary

Registry Version : 1.1
Schema Version   : 1.0
Generated At     : 2026-07-28T09:29:30Z
Categories       : 11
Sources          : 117
Frozen           : True


In [13]:
registry_files = sorted(CONFIG["sources"].rglob("*.json"))

print(f"Total Registry Files : {len(registry_files)}")

Total Registry Files : 117


In [14]:
registry_files = sorted(CONFIG["sources"].rglob("*.json"))

print(f"Total Registry Files : {len(registry_files)}")

Total Registry Files : 117


In [15]:
master_registry = []

for file in registry_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    data["category"] = file.parent.name
    data["registry_file"] = file.name
    data["registry_path"] = str(file.relative_to(CONFIG["sources"]))

    master_registry.append(data)

print(f"Loaded {len(master_registry)} technologies.")

Loaded 117 technologies.


## Registry Manager

Create a reusable interface for querying and managing the source registry throughout the ingestion pipeline.

In [16]:
class RegistryManager:

    def __init__(self, registry):
        self.registry = registry

    def all(self):
        return self.registry

    def total(self):
        return len(self.registry)

    def by_category(self, category):
        return [
            tech for tech in self.registry
            if tech["category"] == category
        ]

    def by_priority(self, priority):
        return [
            tech for tech in self.registry
            if tech["priority"] == priority
        ]

    def ingestable(self):
        return [
            tech for tech in self.registry
            if tech.get("ingestion", False)
        ]

## Using the Registry

In [17]:
registry = RegistryManager(master_registry)

In [18]:
print(f"Total technologies : {registry.total()}")
print(f"Ingestable         : {len(registry.ingestable())}")

Total technologies : 117
Ingestable         : 117


## Download Queue

Collect all documentation sources that will be processed during ingestion.


In [19]:
download_queue = []

for tech in registry.ingestable():

    sources = [
        "official_documentation",
        "github_repository",
        "api_reference",
        "technical_blog",
        "release_notes",
        "research_papers",
        "benchmark_pages",
        "community_resources"
    ]

    for source in sources:
        for url in tech.get(source, []):
            download_queue.append({
                "technology": tech["name"],
                "category": tech["category"],
                "source": source,
                "url": url,
                "id": tech["id"],
                "organization": tech["organization"],
                "license": tech["license"],
                "priority": tech["priority"],
                "update_frequency": tech["update_frequency"],
            })

In [20]:
print(f"Total URLs : {len(download_queue)}")

Total URLs : 291


In [21]:
download_queue[0]

{'technology': 'Document-Aware Chunking',
 'category': 'chunking',
 'source': 'official_documentation',
 'url': 'https://github.com/langchain-ai/langchain/blob/master/README.md',
 'id': 'source-document-aware-chunking',
 'organization': 'LangChain',
 'license': 'MIT',
 'priority': 'high',
 'update_frequency': 'weekly'}

## Document Downloader

Download raw content from the sources listed in the registry.

In [22]:
def download_document(url):

    with httpx.Client(
        timeout=30,
        follow_redirects=True
    ) as client:

        response = client.get(url)
        response.raise_for_status()

    return response.text

In [23]:
sample = download_queue[0]

content = download_document(sample["url"])

print(f"Technology : {sample['technology']}")
print(f"Source     : {sample['source']}")
print(f"Characters : {len(content):,}")

Technology : Document-Aware Chunking
Source     : official_documentation
Characters : 292,163


## Raw Document Storage

Store downloaded documents before processing and chunking.

In [24]:
import json
import re

def save_raw_document(technology, category, source, url, item, content):

    tech = re.sub(r"[^a-zA-Z0-9]+", "_", technology.lower()).strip("_")
    src = re.sub(r"[^a-zA-Z0-9]+", "_", source.lower()).strip("_")

    folder = CONFIG["raw"] / tech
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{src}.json"

    document = {
        "technology": technology,
        "technology_id": item["id"],
        "category": category,
        "organization": item["organization"],
        "license": item["license"],
        "priority": item["priority"],
        "update_frequency": item["update_frequency"],
        "source": source,
        "url": url,
        "content": content,
    }

    with open(path, "w", encoding="utf-8") as file:
        json.dump(document, file, indent=2, ensure_ascii=False)

    return path

In [25]:
path = save_raw_document(
    sample["technology"],
    sample["category"],
    sample["source"],
    sample["url"],
    sample,
    content,
)

print(path)

/content/drive/MyDrive/forge/knowledge_base/raw/document_aware_chunking/official_documentation.json


## Batch Download

Download every source listed in the registry and store it in the raw knowledge base.

In [26]:
download_log = []

for item in download_queue:

    try:
        content = download_document(item["url"])

        path = save_raw_document(
            item["technology"],
            item["category"],
            item["source"],
            item["url"],
            item,
            content,
        )

        download_log.append({
            "technology": item["technology"],
            "source": item["source"],
            "status": "success",
            "characters": len(content),
            "path": str(path)
        })

        print(f"{item['technology']} - {item['source']}")

    except Exception as e:

        download_log.append({
            "technology": item["technology"],
            "source": item["source"],
            "status": "failed",
            "error": str(e)
        })

        print(f"{item['technology']} - {item['source']}")

Document-Aware Chunking - official_documentation
Document-Aware Chunking - github_repository
Fixed-Size Chunking - official_documentation
Fixed-Size Chunking - github_repository
Hierarchical Chunking - official_documentation
Hierarchical Chunking - github_repository
Paragraph-Based Chunking - official_documentation
Paragraph-Based Chunking - github_repository
Parent-Child Chunking - official_documentation
Parent-Child Chunking - github_repository
Recursive Chunking - official_documentation
Recursive Chunking - github_repository
Semantic Chunking - official_documentation
Semantic Chunking - github_repository
Sentence-Based Chunking - official_documentation
Sentence-Based Chunking - github_repository
Sliding Window Chunking - official_documentation
Sliding Window Chunking - github_repository
Token-Based Chunking - official_documentation
Token-Based Chunking - github_repository
Arize AI - official_documentation
AWS SageMaker - official_documentation
Azure AI Foundry - official_documentati

In [27]:
successful = sum(
    log["status"] == "success"
    for log in download_log
)

failed = sum(
    log["status"] == "failed"
    for log in download_log
)

print(f"Successful : {successful}")
print(f"Failed     : {failed}")
print(f"Total      : {len(download_log)}")

Successful : 284
Failed     : 7
Total      : 291


In [28]:
failed_downloads = [
    log for log in download_log
    if log["status"] == "failed"
]

failed_downloads[:10]

[{'technology': 'KServe',
  'source': 'official_documentation',
  'status': 'failed',
  'error': "Client error '404 Not Found' for url 'https://kserve.github.io/website/latest/'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404"},
 {'technology': 'Milvus',
  'source': 'official_documentation',
  'status': 'failed',
  'error': "Client error '403 Forbidden' for url 'https://milvus.io/docs'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403"},
 {'technology': 'Milvus',
  'source': 'api_reference',
  'status': 'failed',
  'error': "Client error '403 Forbidden' for url 'https://milvus.io/api-reference/pymilvus/v2.4.x/About.md'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403"},
 {'technology': 'Milvus',
  'source': 'technical_blog',
  'status': 'failed',
  'error': "Client error '403 Forbidden' for url 'https://milvus.io/blog'\nFor more information check: https://developer

## Download Report

Store the results of the ingestion process for future reference.

In [29]:
print(CONFIG.keys())

dict_keys(['project_root', 'manifest', 'sources', 'schemas', 'knowledge', 'raw', 'processed', 'layer1', 'layer2', 'layer3', 'layer4', 'layer5', 'chunks', 'embeddings', 'logs'])


In [30]:
import pandas as pd

download_report = pd.DataFrame(download_log)

report_path = CONFIG["raw"].parent / "download_report.csv"

download_report.to_csv(report_path, index=False)

print(report_path)

/content/drive/MyDrive/forge/knowledge_base/download_report.csv


In [31]:
total = len(download_log)
successful = sum(log["status"] == "success" for log in download_log)
failed = total - successful

print("=" * 50)
print("FORGE KNOWLEDGE INGESTION COMPLETE")
print("=" * 50)
print(f"Total Sources     : {total}")
print(f"Downloaded        : {successful}")
print(f"Failed            : {failed}")
print(f"Success Rate      : {successful / total:.2%}")
print(f"Raw Documents     : {CONFIG['raw']}")
print(f"Download Report   : {report_path}")

FORGE KNOWLEDGE INGESTION COMPLETE
Total Sources     : 291
Downloaded        : 284
Failed            : 7
Success Rate      : 97.59%
Raw Documents     : /content/drive/MyDrive/forge/knowledge_base/raw
Download Report   : /content/drive/MyDrive/forge/knowledge_base/download_report.csv
